# Radeon-Assistant 启动与功能演示

AMD Radeon Hackathon 2026-07 Track 2 参赛项目

**技术栈**: vLLM + ROCm 7.2.1 + Qwen2.5-14B-Instruct + AMD Radeon PRO W7900

**核心能力**: RAG 知识库 / 工具调用 / 多步骤任务规划 / HITL 审批 / 审计日志

## 第一步：环境配置

设置 ROCm 环境变量（W7900 是 gfx1100 架构，必须 override）

In [ ]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'
os.environ['PYTORCH_ROCM_ARCH'] = 'gfx1100'
os.environ['HIP_VISIBLE_DEVICES'] = '0'
os.environ['HSA_ENABLE_SDMA'] = '0'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
print('✅ ROCm 环境变量已设置')

## 第二步：安装依赖

安装基础依赖 + vLLM ROCm 预编译 wheel

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-r', '../requirements.txt'], capture_output=True, text=True)
print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)

In [ ]:
# 安装 vLLM ROCm wheel
import subprocess
result = subprocess.run(
    ['pip', 'install', 'vllm', '--extra-index-url', 'https://wheels.vllm.ai/rocm/', '--no-cache-dir'],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)

## 第三步：验证 GPU 环境

确认 ROCm 和 W7900 GPU 可用

In [ ]:
import subprocess
result = subprocess.run(['rocm-smi', '--showproductname'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# 验证 PyTorch ROCm
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'ROCm 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 第四步：下载模型（首次运行）

下载 Qwen2.5-14B-Instruct（约 30GB）

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../scripts/download_model.py', '--model', 'qwen2.5-14b'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)

## 第五步：初始化 Agent

加载模型 + 初始化记忆 + 创建 Agent

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
from inference.engine import InferenceEngine, InferenceConfig
from memory.manager import MemoryManager
from agent.core import RadeonAgent

with open('../config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

model_config = config['model']
print('正在加载模型（首次约 2-3 分钟）...')
inference_config = InferenceConfig(
    model_path=model_config['path'],
    n_ctx=model_config['n_ctx'],
    dtype=model_config['dtype'],
    gpu_memory_utilization=model_config['gpu_memory_utilization'],
    tensor_parallel_size=model_config.get('tensor_parallel_size', 1),
)
engine = InferenceEngine(inference_config)
memory_manager = MemoryManager()
agent = RadeonAgent(engine, memory_manager)
print('✅ Agent 初始化完成')

## 第六步：基础对话测试

测试 LLM 推理能力

In [ ]:
response = agent.chat('用 200 字介绍 AMD ROCm', use_rag=False)
print('回复:', response)

## 第七步：工具调用测试（核心功能）

测试 Agent 的工具调用能力

In [ ]:
result = agent.run_task('读取 config.yaml 并告诉我模型路径')
print(f"任务结果: {'✅ 完成' if result['success'] else '❌ 未完成'}")
print(f"评估: {result['reflection'].get('reason', '')}")
print(f"执行步骤数: {len(result['steps'])}")
for i, (s, r) in enumerate(zip(result['steps'], result['results'])):
    tool = s.get('tool') or '无'
    success = '✅' if r.get('success') else '❌'
    print(f"  步骤{i+1}: {s['description'][:40]} | 工具: {tool} {success}")

## 第八步：多步骤任务测试

测试 Agent 的任务规划与执行能力

In [ ]:
result = agent.run_task('在 /tmp 创建 test.txt 文件，写入 hello world，然后读取验证')
print(f"任务结果: {'✅ 完成' if result['success'] else '❌ 未完成'}")
print(f"评估: {result['reflection'].get('reason', '')}")
print(f"执行步骤数: {len(result['steps'])}")
for i, (s, r) in enumerate(zip(result['steps'], result['results'])):
    tool = s.get('tool') or '无'
    success = '✅' if r.get('success') else '❌'
    print(f"  步骤{i+1}: {s['description'][:40]} | 工具: {tool} {success}")

## 第九步：HITL 审批测试

测试高危操作审批机制（自动拒绝回调，演示用）

In [ ]:
# 自动拒绝审批的回调（演示用，实际使用时会弹出 input 提示）
def auto_reject(tool_name, arguments, description):
    print(f"⚠️ 高危操作被自动拒绝: {tool_name}({arguments})")
    return False

# 用 auto_reject 回调创建新 Agent
agent_with_reject = RadeonAgent(engine, memory_manager, approval_callback=auto_reject)
result = agent_with_reject.run_task('删除 /tmp/test.txt')
print(f"任务结果: {'✅ 完成' if result['success'] else '❌ 未完成'}")
print(f"评估: {result['reflection'].get('reason', '')}")

## 第十步：审计日志查看

查看所有操作的审计记录（JSON Lines 格式）

In [ ]:
import json
import os

log_path = '../logs/audit.log'
if not os.path.exists(log_path):
    log_path = './logs/audit.log'

if os.path.exists(log_path):
    with open(log_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                event = rec.get('event', '')
                ts = rec.get('timestamp', '')[:19]
                if event == 'tool_call':
                    success = '成功' if rec.get('success') else '失败'
                    print(f"[{ts}] 工具调用: {rec['tool']}({rec['arguments']}) -> {success}")
                elif event == 'approval':
                    approved = '批准' if rec.get('approved') else '拒绝'
                    print(f"[{ts}] 审批: {rec['tool']} -> {approved}")
                elif event == 'task':
                    success = '完成' if rec.get('success') else '未完成'
                    print(f"[{ts}] 任务: {rec['task'][:30]}... -> {success}")
                elif event == 'chat':
                    print(f"[{ts}] 对话: 消息{rec['message_length']}字 -> 回复{rec['response_length']}字")
            except:
                pass
else:
    print('暂无审计日志')

## 第十一步：性能测试

测试推理速度

In [ ]:
import time
t0 = time.time()
response = engine.generate('请用 100 字介绍 AMD Radeon GPU', max_tokens=100)
t1 = time.time()
print(f'生成内容: {response}')
print(f'\n--- 性能数据 ---')
print(f'总耗时: {t1-t0:.2f}s')
print(f'生成速度: {100/(t1-t0):.1f} tokens/s')

## 第十二步：启动 Web UI（可选）

启动 Streamlit Web 界面（后台运行）

In [ ]:
import subprocess
import threading

def run_streamlit():
    env = os.environ.copy()
    env['HSA_OVERRIDE_GFX_VERSION'] = '11.0.0'
    env['HIP_VISIBLE_DEVICES'] = '0'
    subprocess.run(
        ['streamlit', 'run', '../ui/web_app.py', '--server.port', '7860', '--server.address', '0.0.0.0', '--server.headless', 'true'],
        env=env
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
print('✅ Streamlit 已启动（后台运行）')
print('访问地址: http://localhost:7860 或通过 Jupyter 代理 /proxy/7860/')

## 停止应用

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
print('✅ 应用已停止')